# MotionEmotion — Train on your acted clips (cloud)

Upload recorded clips to Colab, train the model there, and download the checkpoint
and browser model. Select a GPU runtime if one is available.

**First:** `Runtime → Change runtime type → T4 GPU → Save`.


## 1. Install dependencies


In [ ]:
!pip -q install mediapipe==0.10.14 onnx > /dev/null
import torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')


## 2. Upload the pipeline code

Left panel → folder icon → upload these files from `pipeline/`:
`acted_label.py, extract_poses.py, dataset.py, model.py, train.py, infer.py`


In [ ]:
import os
need = ['acted_label.py','extract_poses.py','dataset.py','model.py','train.py','infer.py']
miss = [f for f in need if not os.path.exists(f)]
print('MISSING — upload these:', miss) if miss else print('All pipeline files present.')


## 3. Upload your recorded clips

Zip the folder of clips you recorded with `record.html` (all the `happy_*.webm`, `sad_*.webm`, …),
upload the zip here, and unzip:


In [ ]:
from google.colab import files
up = files.upload()                       # pick your clips.zip
import zipfile, os
z = next(iter(up))
os.makedirs('clips', exist_ok=True)
zipfile.ZipFile(z).extractall('clips')
print('extracted to clips/ :', sum(len(f) for _,_,f in os.walk('clips')), 'files')


## 4. Label, extract poses, train


In [ ]:
!python acted_label.py --videos clips --out data_acted --copy


In [ ]:
!python extract_poses.py --videos data_acted/videos --out data_acted/poses


In [ ]:
# Train for 80 epochs; check validation results for overfitting.
!python train.py --poses data_acted/poses --labels data_acted/labels --epochs 80


## 5. Export & download the model


In [ ]:
!python infer.py --ckpt checkpoints/best.pt --export-onnx motionemotion.onnx
from google.colab import files
files.download('checkpoints/best.pt')
files.download('motionemotion.onnx')


## Tips

- If val accuracy is low, record more clips (aim 15–20 per emotion) and vary poses/people.
- The confusion matrix printed after training shows which emotions get mixed up — merge
  frequently confused classes into fewer classes if needed.
- Keep clips ~5s. Very short clips (<2s) give too few pose windows.
